<a href="https://colab.research.google.com/github/Emelygabrielaa/calgary-building-permits-eda/blob/main/notebooks/01_analysis_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project: Calgary Building Permits, Descriptive Analysis and EDA**

**Path:** Statistician path, Project 1 of 11  
**Author:** Emely Rodriguez  
**Languages:** Python (lead), R (cross-check in a separate notebook)  
**Data as of:** 2026-10-27 snapshot (502,814 rows, 21 columns). The portal showed the data as last updated on 2026-10-27.   
**Licence:** Contains information licensed under the Open Government Licence - City of Calgary.

## Project phases

| Phase | Content | Status |
|---|---|---|
| 0. Data audit | Structure, quality and definitions of the source data | Done |
| **1. Analysis dataset** | **Handling of audit issues, selected columns, derived variables, dated snapshot** | **This notebook** |
| 2. Analysis | Part A (descriptive) and Part B (formal tests and models), one notebook per question | Pending |
| 3. Report | Findings and conclusions | Pending |

## What this project is about
The City of Calgary publishes its building permit application records, which start in June 1999. In this project I use that table to answer five practical questions: how long applications take, how much the projects are worth, how activity changes over time, where it happens, and how complete the data is. In each question's notebook, Part A answers it with descriptive statistics and Part B follows with formal tests and models.

## The five questions

| # |Topic | Question | Part A: descriptive | Part B: going deeper |
|---|---|---|---|---|
| 1 | Processing time | How long do applications take? | Days from application to issue, with percentiles by permit type | Kruskal-Wallis with pairwise tests, and Kaplan-Meier curves |
| 2 | Project cost | How much are the projects worth? | Skewness, log scale, tail, and summaries by class | Lognormal fit check, bootstrap intervals for medians, quantile regression |
| 3 | Volume over time | How does activity change over time? | Counts by year and month, seasonality plots | Decomposition, Mann-Kendall trend test, count regression |
| 4 | Geography | Where does it happen? | Counts and medians by community, and a map | Rates per community and Moran's I |
| 5 | Missingness | How complete is the data? | Missing rates by column and permit type | Chi-square tests and logistic regression of missingness |

## The data

- **Source:** City of Calgary Open Data portal, dataset "Building Permits" (ID c2es-76ed)
- **Provider:** The City of Calgary, Dev Business and Bldg Srvc
- **What a row is:** a building permit application record, identified by `permitnum` (unique, checked in section 0.2.1 of the Phase 0 notebook)
- **Coverage:** applications from 1999-06-22 onward, updated daily
- **Size:** about 503,000 rows and 30 columns in the portal; the snapshot keeps 21 columns (section 1.1.1 of this notebook)
- **Coordinates:** WGS84 (EPSG:4326), in degrees
- **Licence:** City of Calgary Open Data Terms (https://data.calgary.ca/d/Open-Data-Terms/u45n-7awa)


**Reproducibility note:** the portal table is updated daily. Phase 0 queried it directly. Phase 1 downloaded it once, on 2026-10-27, and saved it as a dated snapshot in the project repository. All results from Phase 1 onward are computed on that snapshot, or on the processed dataset built from it, so the values do not change. To rebuild the processed dataset, run section 1.0.1 and then sections 1.2.0 to 1.4.2. Section 1.1 is the one-time download and is not run again.

## Pills for this project
**Board pills:** Descriptive/EDA, Group Comparison, Missing-data analysis, Trend analysis,
Regression analysis, Uncertainty analysis, Spatial

**Method pills:** percentiles, ecdf, skewness, log-transform, heavy-tails, kruskal-wallis,
multiple-comparisons, censoring, kaplan-meier, competing-risks, lognormal-fit, bootstrap, quantile-regression, seasonality,
decomposition, mann-kendall, count-regression, rates-per-area, morans-i, choropleth,
missing-rates, chi-square, logistic-regression, mar-vs-mcar

----------

# **Phase 1: Analysis dataset**

**Goal:** turn the audited source table into a fixed analysis dataset:  
- Freeze a dated copy of the data,   
- Decide how each issue found in Phase 0 is handled,   
- Create the derived variables the five questions need.

**Outputs of this phase:**
- A dated snapshot of the source table (21 columns), stored in the project repository
- A record of the handling decision for each issue in the Phase 0 issues log
- A processed analysis dataset with derived variables, stored in the project repository

**Approach:** the snapshot is downloaded once and never changes. All later phases read the snapshot or the processed dataset and do not query the portal again. Every handling rule is written as code, so the processed dataset can be rebuilt from the snapshot at any time.

## 1.0 Setup

This notebook has its own setup, so it can run without the Phase 0 notebook.

### 1.0.1 Libraries and portal address

In [ ]:
# Import the libraries: web requests, data tables, and file handling
import requests, pandas as pd
from io import StringIO
from pathlib import Path

# Address of the dataset's data endpoint (CSV format)
BASE = "https://data.calgary.ca/resource/c2es-76ed.csv"

# Folders where the snapshot and the processed dataset will be stored, relative to the repository root
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# Create the folders if they do not exist yet
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

The folders `data/raw` and `data/processed` are created in the session storage. They stay empty until the snapshot is saved in section 1.1.6.

## 1.1 Snapshot

The snapshot is a dated copy of the source table. It is downloaded once, saved as received, and not cleaned. All handling decisions are made later, in section 1.2.

This section was run once, on 2026-10-27. In later sessions it is not run again, because a new download could return a different table. The notebook continues from section 1.2.0, which loads the stored file.



### 1.1.1 Columns kept

The snapshot keeps 21 of the 30 columns. The nine columns left out are not needed for any of the five questions:

| Left out | Reason |
|---|---|
| `applicantname`, `contractorname` | Personal or business names, not used in any question |
| `description`, `originaladdress` | Free text and street addresses, not used in any question |
| `locationtypes`, `locationaddresses`, `locationswkt`, `locationsgeojson` | Detail of each location in multi-location permits, large in size; `locationcount` and the coordinates are kept instead |
| `point` | Repeats `latitude` and `longitude` |

In [ ]:
# Columns kept in the snapshot, grouped by role
KEEP = [
    # identifier and status
    "permitnum", "statuscurrent",
    # dates
    "applieddate", "issueddate", "completeddate",
    # classification
    "permittype", "permittypemapped", "permitclass", "permitclassgroup", "permitclassmapped",
    "workclass", "workclassgroup", "workclassmapped",
    # size and cost
    "housingunits", "estprojectcost", "totalsqft",
    # location
    "communitycode", "communityname", "latitude", "longitude", "locationcount",
]

# Check that the list has 21 columns and no repeats
print(len(KEEP), len(set(KEEP)))

21 21


The list has 21 columns, with no repeats.

### 1.1.2 Portal total before download

The table is updated daily, so the total is read first. The downloaded file must match it. If the portal changes during the download, the download is repeated.

In [ ]:
# Ask the portal how many rows the table has right now
r = requests.get(BASE, params={"$select": "count(*) as n"}, timeout=120)
r.raise_for_status()                                   # stop with an error if the portal rejects the request
total_before = int(pd.read_csv(StringIO(r.text))["n"][0])  # read the count from the answer

# Show the total
print(f"Rows in the portal before download: {total_before:,}")

Rows in the portal before download: 502,814


The portal table had 502,814 rows at the moment before the download.

### 1.1.3 Download function

The table is downloaded in pages of 50,000 rows, ordered by `permitnum`, so that no row is skipped or repeated between pages. Every column is read as text and the values "N/A" and "NA" are kept as they are, so nothing is changed at this step.

In [ ]:
# Download the kept columns in pages of 50,000 rows and return them as one table of raw text
def download_snapshot(page_size=50_000):
    pages, offset = [], 0
    while True:
        params = {
            "$select": ",".join(KEEP),   # only the 21 kept columns
            "$order": "permitnum",       # fixed order, so pages do not overlap
            "$limit": page_size,         # rows per page
            "$offset": offset,           # where this page starts
        }
        r = requests.get(BASE, params=params, timeout=300)  # ask the portal for one page
        r.raise_for_status()                                # stop with an error if the portal rejects it
        page = pd.read_csv(StringIO(r.text), dtype=str, keep_default_na=False)  # raw text, "N/A" kept
        if page.empty:                                      # no more rows, stop
            break
        pages.append(page)
        offset += page_size
        print(f"Downloaded {offset:,} rows requested")     # simple progress message
    return pd.concat(pages, ignore_index=True)              # join all pages into one table

### 1.1.4 Download and check

The download runs once. Then the number of rows and the number of unique permit numbers are compared with the portal total from 1.1.2.

In [ ]:
# Run the download (takes a few minutes)
snap = download_snapshot()

# Compare the downloaded table with the portal total read before the download
print(f"Rows downloaded:         {len(snap):,}")
print(f"Unique permit numbers:   {snap['permitnum'].nunique():,}")
print(f"Portal total (1.1.2):    {total_before:,}")
print(f"Columns downloaded:      {snap.shape[1]}")

Downloaded 50,000 rows requested
Downloaded 100,000 rows requested
Downloaded 150,000 rows requested
Downloaded 200,000 rows requested
Downloaded 250,000 rows requested
Downloaded 300,000 rows requested
Downloaded 350,000 rows requested
Downloaded 400,000 rows requested
Downloaded 450,000 rows requested
Downloaded 500,000 rows requested
Downloaded 550,000 rows requested
Rows downloaded:         502,814
Unique permit numbers:   502,814
Portal total (1.1.2):    502,814
Columns downloaded:      21


The snapshot has 502,814 rows and 21 columns. The number of unique permit numbers equals the number of rows and both equal the portal total, so no row was skipped or repeated between pages.

### 1.1.5 Portal total after download

The portal total is read again. If it equals the total from 1.1.2, the table did not change during the download and the snapshot is complete.

In [ ]:
# Ask the portal for the row count again, after the download
r = requests.get(BASE, params={"$select": "count(*) as n"}, timeout=120)
r.raise_for_status()                                        # stop with an error if the portal rejects the request
total_after = int(pd.read_csv(StringIO(r.text))["n"][0])    # read the count from the answer

# Compare the totals before and after the download
print(f"Portal total before: {total_before:,}")
print(f"Portal total after:  {total_after:,}")
print("Table unchanged during download:", total_before == total_after)

Portal total before: 502,814
Portal total after:  502,814
Table unchanged during download: True


The portal total did not change during the download. The snapshot is complete and matches the portal as of 2026-10-27.

### 1.1.6 Saving the snapshot

Two file formats are compared by size before choosing one.

In [ ]:
# Save the table in two formats in a temporary folder to compare their sizes
import os
snap.to_csv("test_snapshot.csv.gz", index=False)         # compressed CSV, opens in Python and R
snap.to_parquet("test_snapshot.parquet", index=False)    # Parquet, smaller, needs a library in R

# Show the size of each file in megabytes
for f in ["test_snapshot.csv.gz", "test_snapshot.parquet"]:
    print(f"{f}: {os.path.getsize(f) / 1_000_000:.1f} MB")

test_snapshot.csv.gz: 20.2 MB
test_snapshot.parquet: 23.4 MB


The compressed CSV (20.2 MB) is smaller than the Parquet file (23.4 MB), and both are under the 25 MB limit of the GitHub web upload. The compressed CSV is chosen because Python and R open it directly.

In [ ]:
# Final file name, with the date of the download
SNAP_DATE = "2026-10-27"
SNAP_FILE = RAW_DIR / f"building_permits_{SNAP_DATE}.csv.gz"

# Save the snapshot as compressed CSV
snap.to_csv(SNAP_FILE, index=False)

# Remove the two test files used for the size comparison
os.remove("test_snapshot.csv.gz")
os.remove("test_snapshot.parquet")

# Show the saved file and its size
print(SNAP_FILE, f"{os.path.getsize(SNAP_FILE) / 1_000_000:.1f} MB")

data/raw/building_permits_2026-10-27.csv.gz 20.2 MB


The snapshot is saved as `building_permits_2026-10-27.csv.gz` (20.2 MB) in `data/raw/`, and the two test files were removed.

### 1.1.7 Check of the saved file

The file is read back from disk, as text, and compared with the table in memory. The check also confirms that the raw text values "N/A" were kept and not converted to missing values.

In [ ]:
# Read the saved file back as raw text, keeping "N/A" as it is
check = pd.read_csv(SNAP_FILE, dtype=str, keep_default_na=False)

# Compare the file with the table in memory
print("Same shape:     ", check.shape == snap.shape)
print("Same content:   ", check.equals(snap))

# Count the raw "N/A" values in the community name column of both tables
print("N/A in memory:  ", (snap["communityname"] == "N/A").sum())
print("N/A in the file:", (check["communityname"] == "N/A").sum())

Same shape:      True
Same content:    True
N/A in memory:   921
N/A in the file: 921


The saved file has the same shape and the same content as the table in memory. The 921 raw "N/A" values in `communityname` were kept, matching the count found in Phase 0. The snapshot is stored exactly as received from the portal.

### 1.1.8 Storing the snapshot in the repository

Colab's session storage is temporary. The file is downloaded to the local computer and then uploaded to the repository folder `data/raw/` on GitHub.

In [ ]:
# Download the snapshot file from Colab to the local computer
from google.colab import files
files.download(str(SNAP_FILE))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The snapshot is stored in the project repository at `data/raw/building_permits_2026-10-27.csv.gz`. From this point on, the project works from this file and does not query the portal again.

### 1.1.9 Check of the stored file

The snapshot is read from the repository on GitHub, the same way later notebooks will read it, and compared with the table downloaded from the portal.

In [ ]:
# Address of the snapshot file in the public repository (raw file)
SNAP_URL = ("https://raw.githubusercontent.com/Emelygabrielaa/"
            "calgary-building-permits-eda/main/data/raw/building_permits_2026-10-27.csv.gz")

# Read the file from GitHub as raw text, keeping "N/A" as it is
from_github = pd.read_csv(SNAP_URL, dtype=str, keep_default_na=False)

# Compare it with the table downloaded from the portal
print("Same shape:  ", from_github.shape == snap.shape)
print("Same content:", from_github.equals(snap))

Same shape:   True
Same content: True


The file stored in the repository is identical to the table downloaded from the portal. Section 1.1 is complete: the snapshot is fixed, verified, and stored in the repository.

## 1.2 Handling decisions

Each issue found in the Phase 0 audit receives one decision here, written as code. The original columns of the snapshot are never overwritten: where a value is corrected or excluded, the change goes into a new column or a flag, so the raw value stays available.

| Step | Issue | Rows affected |
|---|---|---|
| 1.2.1 | Data types: text to dates and numbers | all |
| 1.2.2 | Dates out of order | 33 |
| 1.2.3 | Closed applications without an issue date, and Completed without a completion date | 292 and 25 |
| 1.2.4 | Negative estimated project cost | 8 |
| 1.2.5 | Zero and negative `totalsqft` | 654 |
| 1.2.6 | Negative `housingunits` | 2 |
| 1.2.7 | Coordinates equal to zero and community "N/A" | 921 |
| 1.2.8 | Same-day issue (time to issue equal to zero) | 105,304 |

### 1.2.0 Loading the snapshot

The stored snapshot is read from the repository as raw text, exactly as saved. Every session of this notebook starts here.

In [ ]:
# Address of the snapshot file in the public repository (raw file)
SNAP_URL = ("https://raw.githubusercontent.com/Emelygabrielaa/"
            "calgary-building-permits-eda/main/data/raw/building_permits_2026-10-27.csv.gz")

# Read the snapshot as raw text, keeping "N/A" as it is
snap = pd.read_csv(SNAP_URL, dtype=str, keep_default_na=False)

# Check the size: it must be 502,814 rows and 21 columns
print(snap.shape)

(502814, 21)


The snapshot has 502,814 rows and 21 columns, equal to the file stored on 2026-10-27.

### 1.2.1 Data types

The snapshot is all text. A working copy is made, and before converting the numeric and date columns, the values that are not a number or a date are listed. This shows how missing values are written in each column.

In [ ]:
# Working copy of the snapshot, the original table stays untouched
df = snap.copy()

# Columns to convert
DATE_COLS = ["applieddate", "issueddate", "completeddate"]
NUM_COLS = ["housingunits", "estprojectcost", "totalsqft", "locationcount", "latitude", "longitude"]

# For each column, list the text values that cannot be read as a number or a date
for col in NUM_COLS + DATE_COLS:
    values = df[col]
    parsed = pd.to_numeric(values, errors="coerce") if col in NUM_COLS else pd.to_datetime(values, errors="coerce")
    bad = values[parsed.isna()].value_counts()          # values that failed to convert, with their counts
    print(col, "->", bad.to_dict() if len(bad) else "all values convert")

housingunits -> all values convert
estprojectcost -> {'': 39023}
totalsqft -> {'': 346307}
locationcount -> {'': 921}
latitude -> all values convert
longitude -> all values convert
applieddate -> all values convert
issueddate -> {'': 18863}
completeddate -> {'': 19852}


The only values that cannot be converted are empty text values, and their counts equal the missing counts found in Phase 0. No other text appears in the numeric or date columns, so the conversion is safe.

The empty values become missing values and the rest are converted. The conversion has no `errors="coerce"` option, so it stops with an error if any other value cannot be converted.

In [ ]:
# Convert the numeric columns: empty text becomes a missing value, any other bad value raises an error
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col].replace("", pd.NA))

# Convert the date columns in the same way
for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col].replace("", pd.NA))

# Show the type and the number of missing values of each converted column
summary = pd.DataFrame({"type": df[NUM_COLS + DATE_COLS].dtypes.astype(str),
                        "missing": df[NUM_COLS + DATE_COLS].isna().sum()})
print(summary)

                          type  missing
housingunits             int64        0
estprojectcost         float64    39023
totalsqft              float64   346307
locationcount          float64      921
latitude               float64        0
longitude              float64        0
applieddate     datetime64[ns]        0
issueddate      datetime64[ns]    18863
completeddate   datetime64[ns]    19852


The three date columns are now dates and the six numeric columns are numbers. The missing counts equal the Phase 0 audit. The text columns, including the "N/A" in `communityname`, are not changed here.

### 1.2.2 Dates out of order

Phase 0 found applications where one date is earlier than a date that should come before it. The rows are listed here before any decision is made.

In [ ]:
# Flag the three kinds of order problem (comparisons with a missing date are False)
issued_before_applied    = df["issueddate"]    < df["applieddate"]
completed_before_issued  = df["completeddate"] < df["issueddate"]
completed_before_applied = df["completeddate"] < df["applieddate"]

# Number of rows in each case and in any of them
print("Issued before applied:    ", issued_before_applied.sum())
print("Completed before issued:  ", completed_before_issued.sum())
print("Completed before applied: ", completed_before_applied.sum())
any_problem = issued_before_applied | completed_before_issued | completed_before_applied
print("Rows with any problem:    ", any_problem.sum())

# Show the rows, with the size of the gap in days for the completed before issued case
cols = ["permitnum", "statuscurrent", "permittype", "applieddate", "issueddate", "completeddate"]
out = df.loc[any_problem, cols].copy()
out["days_completed_before_issued"] = (df["issueddate"] - df["completeddate"]).dt.days
print(out.sort_values("days_completed_before_issued", ascending=False).to_string(index=False))

Issued before applied:     1
Completed before issued:   31
Completed before applied:  1
Rows with any problem:     33
   permitnum statuscurrent                        permittype applieddate issueddate completeddate  days_completed_before_issued
BP2019-10203     Completed Commercial / Multi Family Project  2019-07-31 2021-05-28    2019-09-06                         630.0
BP2015-01070     Cancelled   Residential Improvement Project  2015-02-02 2016-12-28    2015-09-21                         464.0
BP2015-01071     Cancelled   Residential Improvement Project  2015-02-02 2016-12-19    2015-09-21                         455.0
BP2014-08309     Completed   Residential Improvement Project  2014-06-04 2015-10-22    2015-01-30                         265.0
BP2017-13337     Completed   Residential Improvement Project  2017-10-04 2018-10-26    2018-05-02                         177.0
BP2017-13339     Completed   Residential Improvement Project  2017-10-04 2018-10-26    2018-05-02                 

33 applications (0.007% of the table) have dates in an impossible order. In 31 of them the completion date is earlier than the issue date, with gaps from 1 to 630 days; in one the issue date is earlier than the application date; and in one the completion date is earlier than the application date. The data does not show which of the dates is wrong, so none can be corrected with certainty.

33 applications (0.007% of the table) have dates in an impossible order. In 31 of them the completion date is earlier than the issue date, with gaps from 1 to 630 days; in one the issue date is earlier than the application date; and in one the completion date is earlier than the application date. The data does not show which of the dates is wrong, so none can be corrected with certainty.

**Decision.** The raw dates are kept, the 33 rows are flagged in a new column `date_order_issue`, and durations are left empty only where the order is impossible. The rows are not removed.

**Reasons:**
- The data does not show which of the two dates is wrong. In the 31 rows where completion comes before issue, either date could have been entered or updated incorrectly, so replacing one of them would be a guess.
- The rows are valid in everything else (type, cost, location, application date), and they are only 0.007% of the table. Removing them would lose information for no gain, and keeping them changes no count.
- A flag keeps the decision visible and reversible. In Part B, the main tests can be repeated without the flagged rows to confirm that they do not affect the results.
- Time to issue uses the application and issue dates, so only 1 row is affected. Time to close uses the issue and completion dates, so it is affected only in the rows where the completion date is earlier than the issue date.

In [ ]:
# Classify each row by its date order problem; rows without a problem are labelled "none"
# (the three problems do not overlap, as the counts in the previous cell show)
df["date_order_issue"] = "none"
df.loc[issued_before_applied,    "date_order_issue"] = "issued before applied"
df.loc[completed_before_issued,  "date_order_issue"] = "completed before issued"
df.loc[completed_before_applied, "date_order_issue"] = "completed before applied"

# Check the number of rows in each category
print(df["date_order_issue"].value_counts())

date_order_issue
none                        502781
completed before issued         31
issued before applied            1
completed before applied         1
Name: count, dtype: int64


The flag `date_order_issue` marks 33 rows: 31 with the completion date before the issue date, 1 with the issue date before the application date, and 1 with the completion date before the application date. The other 502,781 rows are labelled "none". The raw dates are unchanged.

### 1.2.3 Missing issue and completion dates

Phase 0 found applications in a closed status without an issue date (292) and Completed applications without a completion date (25). The rows are described here before any decision is made.

In [ ]:
# Closed applications without an issue date (statuses where an issue date is expected)
closed_no_issue = df["statuscurrent"].isin(["Completed", "Expired", "File Closed"]) & df["issueddate"].isna()
print("Closed without issue date:", closed_no_issue.sum())
print(df.loc[closed_no_issue, "statuscurrent"].value_counts().to_string())

# Completed applications without a completion date
completed_no_date = (df["statuscurrent"] == "Completed") & df["completeddate"].isna()
print("\nCompleted without completion date:", completed_no_date.sum())

# Year of application of both groups, to see if the problem is concentrated in some years
years = pd.DataFrame({
    "closed_no_issue": df.loc[closed_no_issue, "applieddate"].dt.year.value_counts(),
    "completed_no_date": df.loc[completed_no_date, "applieddate"].dt.year.value_counts(),
}).fillna(0).astype(int).sort_index()
print("\n", years.to_string())

# Rows with both problems at once
print("\nBoth problems:", (closed_no_issue & completed_no_date).sum())

Closed without issue date: 292
statuscurrent
Completed      194
Expired         85
File Closed     13

Completed without completion date: 25

              closed_no_issue  completed_no_date
applieddate                                    
1999                      11                  0
2000                      11                  0
2001                      19                  0
2002                       7                  0
2003                      14                  0
2004                      11                  0
2005                      20                  0
2006                      34                  0
2007                      32                  0
2008                      22                  0
2009                      15                  1
2010                      20                  1
2011                       5                  0
2012                       3                  0
2013                       3                  4
2014                       3             

292 applications in a closed status have no issue date (Completed 194, Expired 85, File Closed 13), and 25 Completed applications have no completion date. Both problems are concentrated in some years. The missing issue dates are mostly in applications from 1999 to 2010 (216 of 292), with a smaller group in 2023 and 2024 (29). The missing completion dates are mostly in applications from 2013 to 2015 (20 of 25). One application has both problems.

**Decision.** The missing values are kept and not filled. Two flags mark the rows, and the durations that need the missing date are left empty in those rows. The rows are not removed.

**Reasons:**
- A Completed application must have been issued at some point, so for the 194 Completed rows the issue date exists in reality and was not recorded. The data does not contain it, and any estimate would be a date that was never recorded.
- For Expired and File Closed (98 rows) the data does not show whether the permit was ever issued, so no value can be assumed either.
- The rows are valid in everything else (type, cost, location, application date) and are 0.06% of the table. Removing them would lose information for no gain.
- The flags keep the pattern visible. In the missingness question (Q5), they are used to describe the missing dates by year and by status.
- Time to issue is left empty in the flagged closed rows, and time to close is left empty in the flagged Completed rows, so neither duration is calculated from a date that does not exist.

In [ ]:
# Flag closed applications without an issue date, and Completed applications without a completion date
df["issue_date_missing_closed"] = closed_no_issue
df["completion_date_missing_completed"] = completed_no_date

# Check the number of rows in each flag and in both
print("Issue date missing, closed:       ", df["issue_date_missing_closed"].sum())
print("Completion date missing, Completed:", df["completion_date_missing_completed"].sum())
print("Both flags:                        ", (df["issue_date_missing_closed"] & df["completion_date_missing_completed"]).sum())

Issue date missing, closed:        292
Completion date missing, Completed: 25
Both flags:                         1


The flag `issue_date_missing_closed` marks 292 rows and `completion_date_missing_completed` marks 25 rows, with 1 row in both. The missing dates are not filled and the rows are kept.

### 1.2.4 Estimated project cost

Phase 0 found 8 negative values in `estprojectcost`, and no zeros. The negative rows are listed here, together with the smallest positive values, to see if very small amounts are also a problem.

In [ ]:
# Rows with a negative estimated cost
neg = df["estprojectcost"] < 0
print("Negative costs:", neg.sum())
cols = ["permitnum", "statuscurrent", "permittype", "applieddate", "estprojectcost"]
print(df.loc[neg, cols].sort_values("estprojectcost").to_string(index=False))

# How many positive costs are very small
cost = df["estprojectcost"]
for limit in [10, 100, 1_000]:
    print(f"Cost between 0 and {limit:,}:", ((cost > 0) & (cost < limit)).sum())

# The 10 smallest positive costs
print("\nSmallest positive costs:")
print(df.loc[cost > 0, cols].nsmallest(10, "estprojectcost").to_string(index=False))

Negative costs: 8
   permitnum statuscurrent                        permittype applieddate  estprojectcost
BP2004-09155     Completed Commercial / Multi Family Project  2004-07-12   -1.355130e+08
BP2008-05122     Completed Commercial / Multi Family Project  2008-04-30   -1.118600e+07
BP2010-12171     Completed Commercial / Multi Family Project  2010-09-20   -3.500000e+06
BP2008-15062     Completed        Single Construction Permit  2008-12-22   -1.601839e+06
BP2008-03205     Completed   Residential Improvement Project  2008-03-25   -1.624675e+04
BP2011-10352     Completed   Residential Improvement Project  2011-08-18   -1.579772e+04
BP2011-15870     Completed   Residential Improvement Project  2011-12-19   -4.199900e+02
BP2012-04803     Completed   Residential Improvement Project  2012-04-25   -1.600000e+02
Cost between 0 and 10: 509
Cost between 0 and 100: 630
Cost between 0 and 1,000: 5565

Smallest positive costs:
   permitnum statuscurrent                        permittype appliedd

All 8 negative costs are in Completed applications from 2004 to 2012, and they range from -160 to -135,513,000. Separately, very small positive costs are frequent: 509 applications have a cost below 10 (including values of 1.00 and 0.01), 630 below 100, and 5,565 below 1,000. A cost of 1 dollar for a Commercial / Multi Family project is not a real estimate and looks like a placeholder value.

**Decision 1: negative costs.** The raw column `estprojectcost` is kept unchanged. A new column `estprojectcost_clean` is created, where the 8 negative values are set to missing, and a column `cost_issue` marks the rows. The rows are not removed.

**Reasons:**
- A negative cost could be a sign error or an adjustment entered with a minus sign. The data does not show which, and the amounts range from 160 to 135 million, so taking the absolute value would be a guess.
- Setting the values to missing is the only choice that does not invent a number, and the raw value remains in the original column.
- The 8 rows are valid in everything else (type, location, dates) and are 0.002% of the table, so they stay in all other counts. Only the cost analysis (Q2) is affected.
- A separate flag column keeps the decision visible, and it can be extended in the next step for the very small values.

In [ ]:
# Clean cost column: same as the raw cost, but negative values become missing
df["estprojectcost_clean"] = df["estprojectcost"].where(df["estprojectcost"] >= 0)

# Flag column for cost problems; rows without a problem are labelled "none"
df["cost_issue"] = "none"
df.loc[df["estprojectcost"] < 0, "cost_issue"] = "negative"

# Check the result
print(df["cost_issue"].value_counts())
print("Missing in raw column:  ", df["estprojectcost"].isna().sum())
print("Missing in clean column:", df["estprojectcost_clean"].isna().sum())

cost_issue
none        502806
negative         8
Name: count, dtype: int64
Missing in raw column:   39023
Missing in clean column: 39031


The 8 negative costs are set to missing in `estprojectcost_clean` and marked as "negative" in `cost_issue`. The raw column is unchanged. The clean column has 39,031 missing values, the 39,023 of the raw column plus the 8 negatives.

**Decision 2: very small costs.** Phase 0 did not check positive values close to zero. The small values are described here before any decision is made.

In [ ]:
# Positive costs below 1,000, split into ranges
cost = df["estprojectcost_clean"]
ranges = pd.cut(cost[(cost > 0) & (cost < 1_000)], bins=[0, 1, 2, 10, 100, 500, 1_000], right=False)
print(ranges.value_counts().sort_index().to_string())

# Most frequent exact values below 1,000
print("\nMost frequent exact values below 1,000:")
print(cost[(cost > 0) & (cost < 1_000)].value_counts().head(10).to_string())

# Share of each permit type among costs below 10, compared with its share in the whole table
small = df[(cost > 0) & (cost < 10)]
share = pd.DataFrame({
    "below_10": small["permittype"].value_counts(),
    "all_rows": df["permittype"].value_counts(),
}).fillna(0).astype(int)
share["pct_of_type_below_10"] = (100 * share["below_10"] / share["all_rows"]).round(2)
print("\n", share.to_string())

# Year of application of costs below 10
print("\nBy year of application (costs below 10):")
print(small["applieddate"].dt.year.value_counts().sort_index().to_string())

estprojectcost_clean
[0, 1)            1
[1, 2)          499
[2, 10)           9
[10, 100)       121
[100, 500)     1556
[500, 1000)    3379

Most frequent exact values below 1,000:
estprojectcost_clean
500.0    855
1.0      497
800.0    323
100.0    283
600.0    259
900.0    209
300.0    181
200.0    177
700.0    172
400.0    151

                                    below_10  all_rows  pct_of_type_below_10
permittype                                                                 
Commercial / Multi Family Project       201    101057                  0.20
Demolition                                0     17680                  0.00
Environmental Restoration Permit          0       747                  0.00
Residential Improvement Project         308    226823                  0.14
Single Construction Permit                0    156507                  0.00

By year of application (costs below 10):
applieddate
2002      1
2003      1
2004      1
2011      1
2012      1
2014      2
2015   

Costs below 10 appear in 509 applications (0.10% of the table). Of these, 497 are exactly 1.00, which makes a spike: the whole range from 2 to 10 has only 9 applications, and the range from 10 to 100 has 121. These values appear only in Commercial / Multi Family projects (201) and Residential Improvement projects (308), and 62% of them are in applications from 2019 to 2021 (317 of 509). The most frequent values above that range are round amounts such as 500 (855), 800, 100 and 600, which look like real, small estimates and not like placeholders.

**Decision 2: very small costs.** In `estprojectcost_clean`, every cost below 10 is set to missing, and the rows are marked "below 10" in `cost_issue`. The raw column is unchanged and the rows are not removed.

**Reasons:**
- The value 1.00 repeats 497 times, only in two permit types and mostly in 2019 to 2021, which is the pattern of a placeholder and not of a real estimate.
- No real construction project is estimated at less than 10 dollars, so the cut-off covers the placeholders (1.00 and 0.01) and the 9 unrealistic values between 2 and 10. The cut-off is a judgment, and it affects 509 rows (0.10% of the table).
- Values from 10 up are not changed. The round amounts between 100 and 1,000 (500, 800, 600) look like real small estimates. The 121 values between 10 and 100 are ambiguous, so they are kept as recorded and can be checked in Part B.
- The raw value stays in `estprojectcost` and the flag keeps the decision visible and reversible.

In [ ]:
# Set costs below 10 to missing in the clean column (negative values are already missing)
small_cost = (df["estprojectcost"] >= 0) & (df["estprojectcost"] < 10)
df.loc[small_cost, "estprojectcost_clean"] = pd.NA

# Mark the rows in the flag column
df.loc[small_cost, "cost_issue"] = "below 10"

# Check the result
print(df["cost_issue"].value_counts())
print("Missing in raw column:  ", df["estprojectcost"].isna().sum())
print("Missing in clean column:", df["estprojectcost_clean"].isna().sum())

cost_issue
none        502297
below 10       509
negative         8
Name: count, dtype: int64
Missing in raw column:   39023
Missing in clean column: 39540


The clean cost column now has 39,540 missing values: the 39,023 of the raw column, the 8 negative costs and the 509 costs below 10. `cost_issue` marks 8 rows as "negative" and 509 as "below 10". The raw column is unchanged.

### 1.2.5 Total square feet

`totalsqft` is recorded only for Single Construction Permit applications (structural missingness found in Phase 0). Among the recorded values, Phase 0 found 652 zeros and 2 negatives. The rows are described here before any decision is made, together with the smallest and the largest positive values.

In [ ]:
# Recorded values only (the rest are structurally missing)
sqft = df["totalsqft"]
rec = df[sqft.notna()]
print("Recorded values:", len(rec))

# The negative values
cols = ["permitnum", "statuscurrent", "permitclassgroup", "applieddate", "totalsqft"]
print("\nNegative values:")
print(df.loc[sqft < 0, cols].to_string(index=False))

# Zeros: by status and by year of application
zero = sqft == 0
print("\nZeros:", zero.sum())
print(df.loc[zero, "statuscurrent"].value_counts().head(5).to_string())
print(df.loc[zero, "applieddate"].dt.year.value_counts().sort_index().to_string())

# Small positive values, in ranges
print("\nPositive values by range:")
print(pd.cut(sqft[sqft > 0], bins=[0, 1, 10, 100, 500, 1_000, 10_000, 50_000], right=False)
      .value_counts().sort_index().to_string())

# The 10 largest values
print("\nLargest values:")
print(df.loc[sqft.notna(), cols].nlargest(10, "totalsqft").to_string(index=False))

Recorded values: 156507

Negative values:
   permitnum statuscurrent permitclassgroup applieddate  totalsqft
BP2008-15062     Completed    Single Family  2008-12-22   -15313.0
BP2013-17750     Cancelled       Two Family  2013-11-28    -2649.0

Zeros: 652
statuscurrent
Cancelled      467
Completed      176
Expired          4
File Closed      3
New              2
applieddate
1999     56
2000     55
2001     44
2002     51
2003     41
2004     38
2005     44
2006     28
2007    174
2008     29
2009     24
2010     10
2011      6
2012     17
2013     16
2014      9
2015      3
2016      1
2017      2
2022      1
2024      1
2025      1
2026      1

Positive values by range:
totalsqft
[0, 1)                 0
[1, 10)                6
[10, 100)              8
[100, 500)            15
[500, 1000)         1576
[1000, 10000)     154237
[10000, 50000)        11

Largest values:
   permitnum statuscurrent permitclassgroup applieddate  totalsqft
BP2005-03188     Cancelled       Two Family  2005-03

`totalsqft` is recorded in 156,507 applications. Among them there are 2 negative values (-15,313 and -2,649) and 652 zeros. The zeros are mostly in Cancelled applications (467 of 652, 72%), and 648 of the 652 are in applications from 1999 to 2017, with a peak of 174 in 2007. There are also 14 positive values below 100 (6 below 10), which are not possible sizes for a building, and 11 values above 10,000, of which 8 of the 10 largest are in Cancelled applications.

**Decision 3: zeros, negatives and values below 100.** In the new column `totalsqft_clean`, zeros, negative values and values below 100 are set to missing. The column `sqft_issue` marks each case. The raw column is unchanged, the rows are not removed, and the structural missing values (applications that are not Single Construction Permit) are not touched.

**Reasons:**
- A building cannot have zero, negative or less than 100 square feet. These values are impossible, and they would distort averages and models if they entered the analysis.
- The zeros are concentrated in old and Cancelled applications (648 of 652 are from 1999 to 2017), which is the pattern of a value that was not recorded, as with a blank.
- The cut-off at 100 is a judgment and adds only 14 rows to the zeros and negatives. The values between 100 and 500 (15 rows) are not changed.
- Large values are not impossible, so they are not changed here. They are reviewed as outliers in Part A.
- The raw value stays in `totalsqft` and the flag keeps the decision visible and reversible.

In [ ]:
# Clean column: same as the raw one, but values below 100 (zeros and negatives included) become missing
df["totalsqft_clean"] = df["totalsqft"].where(df["totalsqft"] >= 100)

# Flag column; rows without a problem (and structural missing values) are labelled "none"
df["sqft_issue"] = "none"
df.loc[df["totalsqft"] == 0, "sqft_issue"] = "zero"
df.loc[df["totalsqft"] < 0, "sqft_issue"] = "negative"
df.loc[(df["totalsqft"] > 0) & (df["totalsqft"] < 100), "sqft_issue"] = "below 100"

# Check the result
print(df["sqft_issue"].value_counts())
print("Missing in raw column:  ", df["totalsqft"].isna().sum())
print("Missing in clean column:", df["totalsqft_clean"].isna().sum())

sqft_issue
none         502146
zero            652
below 100        14
negative          2
Name: count, dtype: int64
Missing in raw column:   346307
Missing in clean column: 346975


`totalsqft_clean` has 346,975 missing values: the 346,307 of the raw column (structural missingness) plus 668 values set to missing (652 zeros, 14 below 100 and 2 negatives). `sqft_issue` marks them. The raw column is unchanged.

### 1.2.6 Housing units

Phase 0 found 2 negative values in `housingunits`, and a very large share of zeros. The negative rows, the meaning of the zeros and the largest values are described here before any decision is made.

In [ ]:
# The negative values
hu = df["housingunits"]
cols = ["permitnum", "statuscurrent", "permittype", "permitclassmapped", "applieddate", "housingunits"]
print("Negative values:")
print(df.loc[hu < 0, cols].to_string(index=False))

# Share of zeros and of positive values by permit type
by_type = pd.DataFrame({
    "rows": df["permittype"].value_counts(),
    "zero": df.loc[hu == 0, "permittype"].value_counts(),
    "positive": df.loc[hu > 0, "permittype"].value_counts(),
}).fillna(0).astype(int)
by_type["pct_zero"] = (100 * by_type["zero"] / by_type["rows"]).round(1)
print("\nBy permit type:")
print(by_type.to_string())

# Share of zeros by year of application
zero_by_year = (hu == 0).groupby(df["applieddate"].dt.year).mean().mul(100).round(1)
print("\nPercent of zeros by year of application:")
print(zero_by_year.to_string())

# The 10 largest values
print("\nLargest values:")
print(df.nlargest(10, "housingunits")[cols].to_string(index=False))

Negative values:
   permitnum statuscurrent                        permittype permitclassmapped applieddate  housingunits
BP2006-18895     Completed Commercial / Multi Family Project   Non-Residential  2006-11-08            -1
BP2007-10323     Completed Commercial / Multi Family Project       Residential  2007-07-04            -1

By permit type:
                                     rows    zero  positive  pct_zero
permittype                                                           
Commercial / Multi Family Project  101057   90954     10101      90.0
Demolition                          17680   17680         0     100.0
Environmental Restoration Permit      747     747         0     100.0
Residential Improvement Project    226823  201024     25799      88.6
Single Construction Permit         156507    2452    154055       1.6

Percent of zeros by year of application:
applieddate
1999    56.3
2000    55.8
2001    53.3
2002    49.5
2003    52.5
2004    55.5
2005    54.2
2006    57.5
200

`housingunits` is zero in all Demolition and Environmental Restoration permits, in 90.0% of Commercial / Multi Family projects, in 88.6% of Residential Improvement projects, and in only 1.6% of Single Construction Permits (2,452 of 156,507). The field therefore counts the housing units that a permit creates, and zero means "no new housing units" in most cases. The share of zeros by year moves between 50% and 76% without a sudden change, following the mix of permit types. The 2 negative values are -1, in Completed Commercial / Multi Family projects from 2006 and 2007. The largest values (up to 715) are all in Commercial / Multi Family projects and are plausible for large apartment buildings, although two of the ten largest are classified as Non-Residential.

**Decision 4: housing units.** In the new column `housingunits_clean`, the 2 negative values are set to missing. All zeros are kept as recorded. The column `housingunits_issue` marks the 2 negatives and the 2,452 zeros in Single Construction Permit applications ("zero in new construction"). The raw column is unchanged and no row is removed.

**Reasons:**
- Zero is a valid value for most of the table (demolitions, renovations and non-residential projects create no housing units), so setting zeros to missing would remove real information.
- A count of housing units cannot be negative, and with only 2 rows there is no way to know the true value, so the rule used for cost and square feet is applied: impossible values become missing and the raw value stays.
- New construction normally creates at least one unit, so the 2,452 zeros in Single Construction Permit may be values that were not recorded. The data does not prove it, so they are not changed, only flagged, and Part A and Part B can test if they affect the results.
- Large values (up to 715) are plausible for large apartment buildings and are not changed.

In [ ]:
# Clean column: same as the raw one, but negative values become missing
df["housingunits_clean"] = df["housingunits"].where(df["housingunits"] >= 0).astype("Int64")

# Flag column; rows without a problem are labelled "none"
df["housingunits_issue"] = "none"
df.loc[df["housingunits"] < 0, "housingunits_issue"] = "negative"
zero_new = (df["housingunits"] == 0) & (df["permittype"] == "Single Construction Permit")
df.loc[zero_new, "housingunits_issue"] = "zero in new construction"

# Check the result
print(df["housingunits_issue"].value_counts())
print("Missing in clean column:", df["housingunits_clean"].isna().sum())

housingunits_issue
none                        500360
zero in new construction      2452
negative                         2
Name: count, dtype: int64
Missing in clean column: 2


`housingunits_clean` has 2 missing values (the 2 negatives). `housingunits_issue` marks 2 rows as "negative" and 2,452 as "zero in new construction". Zeros are kept as recorded and the raw column is unchanged.

### 1.2.7 Location: coordinates and community

Phase 0 found 921 applications with latitude and longitude equal to zero, 921 with the community name "N/A", and 921 with a missing `locationcount`. The rows are described here before any decision is made.

In [ ]:
# The three location problems, as separate conditions
zero_coord = (df["latitude"] == 0) & (df["longitude"] == 0)
na_comm    = df["communityname"] == "N/A"
no_count   = df["locationcount"].isna()
print("Coordinates zero:   ", zero_coord.sum())
print("Community N/A:      ", na_comm.sum())
print("Location count missing:", no_count.sum())
print("All three together: ", (zero_coord & na_comm & no_count).sum())
print("Any of the three:   ", (zero_coord | na_comm | no_count).sum())

# What these rows look like
bad = zero_coord | na_comm | no_count
print("\nCommunity code in these rows:", df.loc[bad, "communitycode"].value_counts().head(3).to_dict())
print("\nBy status:")
print(df.loc[bad, "statuscurrent"].value_counts().head(6).to_string())
print("\nBy permit type:")
print(df.loc[bad, "permittype"].value_counts().to_string())
print("\nBy year of application:")
print(df.loc[bad, "applieddate"].dt.year.value_counts().sort_index().to_string())

# Coordinate range of the rest of the table, to check that no other value is outside Calgary
ok = df[~bad]
print("\nLatitude range:", ok["latitude"].min(), ok["latitude"].max())
print("Longitude range:", ok["longitude"].min(), ok["longitude"].max())

Coordinates zero:    921
Community N/A:       921
Location count missing: 921
All three together:  921
Any of the three:    921

Community code in these rows: {'N/A': 921}

By status:
statuscurrent
Cancelled             889
New                    19
Completed               8
Pre Backfill Phase      3
Issued Permit           2

By permit type:
permittype
Residential Improvement Project      380
Single Construction Permit           268
Commercial / Multi Family Project    224
Demolition                            43
Environmental Restoration Permit       6

By year of application:
applieddate
1999    59
2000    76
2001    45
2002    46
2003    47
2004    43
2005    73
2006    49
2007    52
2008    65
2009    27
2010    16
2011    21
2012    40
2013    74
2014    26
2015    47
2016    12
2017    28
2018    15
2019    11
2020     5
2021     2
2022     9
2023     7
2024     9
2025     2
2026    15

Latitude range: 50.84855413158354 51.21235690717057
Longitude range: -114.3041724255555 -113.

The same 921 applications (0.18% of the table) have latitude and longitude equal to zero, the community name and community code "N/A", and no location count. No other row has a problem: the coordinates of the rest of the table are inside Calgary (latitude 50.85 to 51.21, longitude -114.30 to -113.86). Of the 921 applications, 889 are Cancelled (96.5%), and the others are New (19), Completed (8), Pre Backfill Phase (3) and Issued Permit (2). They appear in all permit types and in every year, with 40 to 76 per year in many years before 2014 and 28 or fewer per year from 2016.

**Decision 5: applications without a location.** In new clean columns, the coordinates, the community name and the community code are set to missing for the 921 applications without a location. The column `location_missing` marks them. The raw columns are unchanged and no row is removed.

**Reasons:**
- Zero coordinates are impossible for Calgary: they would place 921 points in the Atlantic Ocean and would break any map, distance or spatial test if they stayed in the clean columns.
- "N/A" is not a community. Setting it to missing keeps the tables by community correct, and the number of applications without a location is still available from the flag.
- The rows are valid in everything else (type, cost, dates) and are 0.18% of the table, so they stay in all other analyses. Only the geography question (Q4) loses them.
- 96.5% of these applications are Cancelled, so the missing location is related to the status. The flag lets the missingness question (Q5) describe this pattern.
- The raw columns stay in the dataset, so the decision is visible and reversible.

In [ ]:
# Flag the applications without a location (the three problems coincide in the same 921 rows)
df["location_missing"] = (df["latitude"] == 0) & (df["longitude"] == 0)

# Clean columns: same as the raw ones, but missing where the location is missing
for col in ["latitude", "longitude", "communityname", "communitycode"]:
    df[col + "_clean"] = df[col].where(~df["location_missing"])

# Check the result
print("Rows without location:", df["location_missing"].sum())
print(df[["latitude_clean", "longitude_clean", "communityname_clean", "communitycode_clean"]].isna().sum())
print("N/A left in clean community name:", (df["communityname_clean"] == "N/A").sum())

Rows without location: 921
latitude_clean         921
longitude_clean        921
communityname_clean    921
communitycode_clean    921
dtype: int64
N/A left in clean community name: 0


The column `location_missing` marks 921 applications. In the four clean columns (`latitude_clean`, `longitude_clean`, `communityname_clean`, `communitycode_clean`) these rows are missing, and no "N/A" is left in the clean community name. The raw columns are unchanged.

### 1.2.8 Same-day issue

Phase 0 found 105,304 applications issued on the same day as they were applied for. A time to issue of zero days is a valid value, but the size of this group changes how the time to issue must be analysed. The group is described here before any decision is made.

In [ ]:
# Time to issue in days, only for rows with both dates in a valid order (temporary, for this check)
valid = df["issueddate"].notna() & (df["issueddate"] >= df["applieddate"])
d = (df["issueddate"] - df["applieddate"]).dt.days.where(valid)
print("Rows with a valid time to issue:", d.notna().sum())
print("Same day (0 days):", (d == 0).sum(), f"({100 * (d == 0).sum() / d.notna().sum():.1f}% of valid)")

# Distribution of the time to issue
print("\nPercentiles of days to issue:")
print(d.quantile([0.25, 0.5, 0.75, 0.9, 0.99]).to_string())

# Share of same-day issues by permit type
tab = pd.DataFrame({"valid": d.groupby(df["permittype"]).count(),
                    "same_day": (d == 0).groupby(df["permittype"]).sum()})
tab["pct_same_day"] = (100 * tab["same_day"] / tab["valid"]).round(1)
print("\nBy permit type:")
print(tab.to_string())

# Share of same-day issues by year of application
by_year = ((d == 0).groupby(df["applieddate"].dt.year).sum() / d.groupby(df["applieddate"].dt.year).count() * 100).round(1)
print("\nPercent same day, by year of application:")
print(by_year.to_string())

Rows with a valid time to issue: 483950
Same day (0 days): 105304 (21.8% of valid)

Percentiles of days to issue:
0.25      1.0
0.50     14.0
0.75     28.0
0.90     49.0
0.99    204.0

By permit type:
                                    valid  same_day  pct_same_day
permittype                                                       
Commercial / Multi Family Project   95189       981           1.0
Demolition                          16908       819           4.8
Environmental Restoration Permit      617         4           0.6
Residential Improvement Project    216863    103290          47.6
Single Construction Permit         154373       210           0.1

Percent same day, by year of application:
applieddate
1999    24.3
2000    24.9
2001    25.7
2002    25.6
2003    26.6
2004    28.8
2005    27.1
2006    29.7
2007    36.8
2008    41.4
2009    37.4
2010    34.1
2011    33.3
2012    31.6
2013    31.5
2014    32.7
2015    39.6
2016    37.7
2017    29.7
2018    15.6
2019    12.6
2020     

483,950 applications have a valid time to issue, and 105,304 of them (21.8%) were issued on the same day. The median time to issue is 14 days, the 75th percentile is 28 days and the 99th percentile is 204 days. The same-day issues are almost all in Residential Improvement projects: 47.6% of them are issued on the same day, against 0.1% to 4.8% in the other permit types, and Residential Improvement accounts for 103,290 of the 105,304 same-day issues (98%). The share by year of application was between 24% and 41% from 1999 to 2017, fell to 15.6% in 2018, 12.6% in 2019 and 8.4% in 2020, and has been about 1% since 2021.

**Decision 6: same-day issues.** All values are kept as recorded and nothing is changed in the data. The flag `issued_same_day` is created together with the time to issue in section 1.3.

**Reasons:**
- A time to issue of zero days is a valid value. The issue date is recorded, and 98% of these cases are Residential Improvement projects, where a fast process is expected.
- Removing them would remove most of the issued Residential Improvement projects and would make the process look slower than it is. Treating zero as missing would be wrong, because the date exists.
- The share of same-day issues changed sharply over time (24% to 41% until 2017, then about 1% since 2021), so the time to issue cannot be pooled across all years as if it came from one process.
- Part A and Part B report the share of same-day issues separately, and analyse the days to issue by permit type and by period.

### 1.2.9 Summary of the handling decisions

Every raw column is unchanged. Where a value is set to missing, the result goes into a new clean column, and each decision is marked by a flag column.

| Step | Issue | Decision | New columns | Rows affected |
|---|---|---|---|---|
| 1.2.2 | Dates out of order | Raw dates kept, rows flagged, durations left empty where the order is impossible | `date_order_issue` | 33 |
| 1.2.3 | Missing issue or completion date | Missing values kept, rows flagged, durations left empty | `issue_date_missing_closed`, `completion_date_missing_completed` | 292 and 25 (1 in both) |
| 1.2.4 | Negative and very small costs | Set to missing in the clean column, rows flagged | `estprojectcost_clean`, `cost_issue` | 8 negative and 509 below 10 |
| 1.2.5 | Zero, negative and very small `totalsqft` | Set to missing in the clean column, rows flagged | `totalsqft_clean`, `sqft_issue` | 668 (652 zero, 14 below 100, 2 negative) |
| 1.2.6 | Negative `housingunits`, zeros in new construction | Negatives set to missing, zeros kept and flagged | `housingunits_clean`, `housingunits_issue` | 2 negative and 2,452 flagged zeros |
| 1.2.7 | Coordinates zero and community "N/A" | Set to missing in the clean columns, rows flagged | `location_missing`, `latitude_clean`, `longitude_clean`, `communityname_clean`, `communitycode_clean` | 921 |
| 1.2.8 | Same-day issue | No change, flag created in 1.3 | none yet | 105,304 |
  
No row was removed in any step.

In [ ]:
# Check the working table: the 21 snapshot columns plus the new columns created in section 1.2
new_cols = [c for c in df.columns if c not in snap.columns]
print("Shape:", df.shape)
print("New columns:", len(new_cols))
print(new_cols)

Shape: (502814, 35)
New columns: 14
['date_order_issue', 'issue_date_missing_closed', 'completion_date_missing_completed', 'estprojectcost_clean', 'cost_issue', 'totalsqft_clean', 'sqft_issue', 'housingunits_clean', 'housingunits_issue', 'location_missing', 'latitude_clean', 'longitude_clean', 'communityname_clean', 'communitycode_clean']


## 1.3 Derived variables

The derived variables are created from the working table. Each one is defined once, here, so all later phases use the same definitions.

| Step | Variable | Used in |
|---|---|---|
| 1.3.1 | `status_group`: the 24 statuses in six groups | Q1, Q3, Q5 |
| 1.3.2 | `days_to_issue` and `issued_same_day` | Q1 |
| 1.3.3 | `days_to_close` | Q1 |
| 1.3.4 | Year, month and period of application, and the analysis window | Q3 |
| 1.3.5 | Classification level for each question | Q2, Q3, Q4 |

### 1.3.1a Status groups

The City does not publish definitions for the 24 values of `statuscurrent`. Each status is placed in a group according to what its dates show in the Phase 0 audit (section 0.14), and not according to an official meaning.

| Group | Statuses |
|---|---|
| Completed | Completed |
| Ended without issue | Cancelled, Refused |
| Issued, in progress | Issued Permit, Pre Backfill Phase, Pre Board Phase, Final Phase, Re-instated, Work Restarted |
| Closed after issue | Expired, File Closed, Suspended-Expired |
| Open before issue | Pending Plans Review Assignment, In Review, New, Pending Plans Review, Application Received, Pending Prescreen Review, Pending Completeness Review, Pending - Building By-Law Review |
| On hold | Hold, Hold - Prescreen Review, Hold - Pending Plans Review, Hold - Completeness Review |

**Decision.** Six groups are created in the new column `status_group`. The original `statuscurrent` is kept unchanged.

**Reasons:**
- Each group answers a question needed in the analysis: whether the application ended, whether it was issued, and whether it is still in progress.
- Five statuses are not defined by the City (Pre Backfill Phase, Pre Board Phase, Final Phase, Re-instated and Work Restarted). All of them have an issue date in 100% of their rows and almost no completion date, so by the data they behave like issued applications still in progress. A separate group would have the same date pattern and would add a category without adding information.
- The placement is a decision based on dates and not on an official definition. Because `statuscurrent` is kept, anyone can check or change the mapping.
- Rows whose dates contradict their group (for example, 68 In Review and 112 Hold with an issue date) are not handled here. They are a separate decision in the next step.

In [ ]:
# Statuses in each group (the grouping is decided from the dates, see the text above)
STATUS_GROUPS = {
    "Completed": ["Completed"],
    "Cancelled or refused": ["Cancelled", "Refused"],
    "Issued, in progress": ["Issued Permit", "Pre Backfill Phase", "Pre Board Phase",
                            "Final Phase", "Re-instated", "Work Restarted"],
    "Closed after issue": ["Expired", "File Closed", "Suspended-Expired"],
    "Open before issue": ["Pending Plans Review Assignment", "In Review", "New", "Pending Plans Review",
                          "Application Received", "Pending Prescreen Review",
                          "Pending Completeness Review", "Pending - Building By-Law Review"],
    "On hold": ["Hold", "Hold - Prescreen Review", "Hold - Pending Plans Review", "Hold - Completeness Review"],
}

# Turn it into a status -> group lookup, and check that each status appears only once
status_to_group = {s: g for g, statuses in STATUS_GROUPS.items() for s in statuses}
print("Statuses in the mapping:", len(status_to_group))

# Check that every status in the table is mapped (stops with an error if one is missing)
unmapped = set(df["statuscurrent"].unique()) - set(status_to_group)
assert not unmapped, f"Statuses without a group: {unmapped}"

# Create the column and show the rows in each group
df["status_group"] = df["statuscurrent"].map(status_to_group)
print(df["status_group"].value_counts().to_string())
print("Rows without group:", df["status_group"].isna().sum())

Statuses in the mapping: 24
status_group
Completed               455111
Cancelled or refused     15760
Issued, in progress      14277
Closed after issue       12079
Open before issue         3552
On hold                   2035
Rows without group: 0


The 24 statuses are placed in six groups and no row is left without a group. The groups are: Completed (455,111), Cancelled or refused (15,760), Issued, in progress (14,277), Closed after issue (12,079), Open before issue (3,552) and On hold (2,035). The original `statuscurrent` is unchanged.

### 1.3.1b Dates that contradict the status group

Phase 0 found applications whose dates do not fit their status (for example, In Review with an issue date). The dates present in each group are counted here before any decision is made.

In [ ]:
# Which of the two dates is present, by status group
has_issue = df["issueddate"].notna().map({True: "issue date", False: "no issue date"})
has_comp  = df["completeddate"].notna().map({True: "completion date", False: "no completion date"})
tab = pd.crosstab(df["status_group"], [has_issue, has_comp])
print(tab.to_string())

# Detail for the groups with few expected dates: the statuses with a contradiction
open_or_hold = df["status_group"].isin(["Open before issue", "On hold"])
print("\nOpen before issue or On hold, with an issue date:  ", (open_or_hold & df["issueddate"].notna()).sum())
print("Open before issue or On hold, with a completion date:", (open_or_hold & df["completeddate"].notna()).sum())
print(df.loc[open_or_hold & df["issueddate"].notna(), "statuscurrent"].value_counts().to_string())

issueddate               issue date                      no issue date                   
completeddate       completion date no completion date completion date no completion date
status_group                                                                             
Closed after issue            11979                  2              98                  0
Completed                    454893                 24             193                  1
Ended without issue            2590                  2           13155                 13
Issued, in progress              19              14258               0                  0
On hold                          10                102               1               1922
Open before issue                23                 49               1               3479

Open before issue or On hold, with an issue date:   184
Open before issue or On hold, with a completion date: 35
statuscurrent
Hold                        112
In Review                    6

Most applications have the dates that their group expects. Three groups contain contradictions. In Open before issue and On hold, 184 applications have an issue date (Hold 112, In Review 68, Pending Plans Review 3, Pending Prescreen Review 1) and 35 have a completion date. In Issued, in progress, 19 applications (Issued Permit) have a completion date. In Ended without issue, 2,592 of 15,760 applications (16.4%) have an issue date: they were issued and later cancelled or refused.

The group of Cancelled and Refused applications is named "Cancelled or refused" and not "without issue", because 2,592 of its 15,760 applications (16.4%) have an issue date: they were issued and later cancelled or refused.

**Decision 7: dates that contradict the status group.** The group is kept as the recorded status says. The new column `status_date_conflict` marks 205 applications whose dates contradict it. No row is removed and no status is changed.

**Reasons:**
- A status "before issue" or "on hold" with an issue date, or an issued status with a completion date, is a contradiction. The data does not show whether the status or the date is wrong, so moving the row to another group would override the recorded status on a guess.
- The flagged rows are 0.04% of the table, so they can stay in all analyses. Part B can repeat the main tests without them to check that they do not change the results.
- Cancelled or refused applications with an issue date are not a contradiction: an application can be issued and cancelled later. They are not flagged.
- The 98 Closed after issue without an issue date, the 193 Completed without an issue date and the 25 Completed without a completion date were flagged in 1.2.3.

In [ ]:
# Flag applications whose dates contradict their status group
pre_issue = df["status_group"].isin(["Open before issue", "On hold"])
df["status_date_conflict"] = "none"
df.loc[pre_issue & df["completeddate"].notna(), "status_date_conflict"] = "completion date in open or hold status"
df.loc[pre_issue & df["issueddate"].notna(), "status_date_conflict"] = "issue date in open or hold status"
issued_group = df["status_group"] == "Issued, in progress"
df.loc[issued_group & df["completeddate"].notna(), "status_date_conflict"] = "completion date in issued status"

# Check the result, and the group names after the rename
print(df["status_date_conflict"].value_counts().to_string())
print()
print(df["status_group"].value_counts().to_string())

status_date_conflict
none                                      502609
issue date in open or hold status            184
completion date in issued status              19
completion date in open or hold status         2

status_group
Completed              455111
Ended without issue     15760
Issued, in progress     14277
Closed after issue      12079
Open before issue        3552
On hold                  2035


The column `status_date_conflict` marks 205 applications: 184 with an issue date in an open or hold status, 19 with a completion date in an issued status, and 2 with a completion date in an open or hold status. The other 502,609 applications are labelled "none". The status groups and the dates are unchanged.

### 1.3.2 Time to issue

`days_to_issue` is the number of days from the application date to the issue date. It is counted in whole days, because the dates are stored at midnight. Following the decisions in section 1.2, it is empty when:

- the application has no issue date (this includes the 292 closed applications flagged in 1.2.3), or
- the issue date is earlier than the application date (1 application, flagged in 1.2.2).

`issued_same_day` is True when `days_to_issue` is 0, False when it is greater than 0, and empty when `days_to_issue` is empty. Applications issued on the same day are kept as recorded (decision in 1.2.8).

In [ ]:
# Time to issue in days: only where an issue date exists and is not before the application date
valid = df["issueddate"].notna() & (df["issueddate"] >= df["applieddate"])
df["days_to_issue"] = (df["issueddate"] - df["applieddate"]).dt.days.where(valid)

# Same-day flag: True if issued the same day, False if later, empty when there is no time to issue
df["issued_same_day"] = (df["days_to_issue"] == 0).astype("boolean").where(df["days_to_issue"].notna())

# Check the result
print("With a time to issue:   ", df["days_to_issue"].notna().sum())
print("Without a time to issue:", df["days_to_issue"].isna().sum())
print(df["days_to_issue"].describe().round(1).to_string())
print()
print(df["issued_same_day"].value_counts(dropna=False).to_string())

With a time to issue:    483950
Without a time to issue: 18864
count    483950.0
mean         24.0
std          51.7
min           0.0
25%           1.0
50%          14.0
75%          28.0
max        4684.0

issued_same_day
False    378646
True     105304
<NA>      18864


483,950 applications have a time to issue and 18,864 do not (18,863 without an issue date and 1 issued before it was applied for). The mean is 24.0 days and the median is 14 days, with 25% of applications issued in 1 day or less and 75% in 28 days or less. 105,304 applications were issued on the same day. The longest time to issue is 4,684 days (about 12.8 years), which is reviewed as an outlier in Part A.

### 1.3.3 Time to close

`days_to_close` is the number of days from the issue date to the completion date, counted in whole days. In this table `completeddate` is the date on which the application was closed, whatever the outcome (Completed, Cancelled, Refused, Expired and others all carry it), so `days_to_close` measures the time from issue to the end of the application. Following the decisions in section 1.2, it is empty when:

- the application has no issue date or no completion date (including the flagged rows of 1.2.3), or
- the completion date is earlier than the issue date (31 applications, flagged in 1.2.2).

In [ ]:
# Time to close in days: only where both dates exist and the completion is not before the issue
valid = df["issueddate"].notna() & df["completeddate"].notna() & (df["completeddate"] >= df["issueddate"])
df["days_to_close"] = (df["completeddate"] - df["issueddate"]).dt.days.where(valid)

# Check the result, overall and for the Completed group
print("With a time to close:   ", df["days_to_close"].notna().sum())
print("Without a time to close:", df["days_to_close"].isna().sum())
print(df["days_to_close"].describe().round(1).to_string())
print("\nCompleted group:")
print(df.loc[df["status_group"] == "Completed", "days_to_close"].describe().round(1).to_string())

With a time to close:    469483
Without a time to close: 33331
count    469483.0
mean        252.2
std         325.1
min           0.0
25%          64.0
50%         160.0
75%         292.0
max        6768.0

Completed group:
count    454868.0
mean        238.2
std         299.3
min           0.0
25%          62.0
50%         157.0
75%         279.0
max        6768.0


469,483 applications have a time to close and 33,331 do not (no issue date, no completion date, or a completion date earlier than the issue date). The mean is 252.2 days and the median is 160 days, with 75% of applications closed within 292 days. For the Completed group the median is 157 days. The longest time to close is 6,768 days (about 18.5 years), which is reviewed as an outlier in Part A.

### 1.3.4 Time variables and complete periods

The dates of application run from 1999-06-22 to 2026-10-26, so 1999 and 2026 are partial years, and June 1999 and October 2026 are partial months. Annual and monthly counts would show a false drop at both ends if these periods were treated as complete.

**Decision.** All rows are kept. Three time variables and two flags are created:

- `application_year`, `application_month` and `application_yearmonth` (for example "2019-03").
- `full_year`: True for 2000 to 2025, the 26 complete years.
- `full_month`: True from July 1999 to September 2026, the complete months.

**Reasons:**
- The partial years are valid for every other question (time to issue, cost, geography, missingness), so removing them would lose valid applications to solve a problem that only affects the volume question (Q3).
- The flags make the choice visible: Q3 filters with them, and the annual counts use the 26 complete years.

In [ ]:
# Year, month and year-month of application
df["application_year"] = df["applieddate"].dt.year
df["application_month"] = df["applieddate"].dt.month
ym = df["applieddate"].dt.to_period("M")
df["application_yearmonth"] = ym.astype(str)           # text such as "2019-03"

# Flags for complete periods
df["full_year"] = df["application_year"].between(2000, 2025)
df["full_month"] = (ym >= pd.Period("1999-07")) & (ym <= pd.Period("2026-09"))

# Check the result
print("First and last application date:", df["applieddate"].min().date(), df["applieddate"].max().date())
print("\nfull_year:")
print(df["full_year"].value_counts().to_string())
print("\nRows in partial years, by year:")
print(df.loc[~df["full_year"], "application_year"].value_counts().sort_index().to_string())
print("\nRows in partial months, by month:")
print(df.loc[~df["full_month"], "application_yearmonth"].value_counts().sort_index().to_string())

First and last application date: 1999-06-22 2026-10-26

full_year:
full_year
True     475411
False     27403

Rows in partial years, by year:
application_year
1999     6991
2026    20412

Rows in partial months, by month:
application_yearmonth
1999-06     390
2026-10    1699


The dates of application run from 1999-06-22 to 2026-10-26. 475,411 applications are in the 26 complete years (2000 to 2025) and 27,403 are in the partial years: 6,991 in 1999 (from 22 June) and 20,412 in 2026 (to 26 October). Two months are partial: June 1999 (390 applications) and October 2026 (1,699 applications). All rows are kept, and the flags `full_year` and `full_month` mark the complete periods.

### 1.3.5 Classification level for each question

The table has classification columns at different levels of detail. Phase 0 proposed which level to use in each question. The values of the columns are listed here, with the blanks and the undefined values, before the choices are written.

In [ ]:
# Values of the classification columns that will be used (blanks are shown as '')
for col in ["permittype", "permittypemapped", "permitclassmapped", "workclassmapped"]:
    print(col)
    print(df[col].value_counts(dropna=False).to_string())
    print()

# The 16 values of the grouped permit class
print("permitclassgroup")
print(df["permitclassgroup"].value_counts(dropna=False).to_string())

permittype
permittype
Residential Improvement Project      226823
Single Construction Permit           156507
Commercial / Multi Family Project    101057
Demolition                            17680
Environmental Restoration Permit        747

permittypemapped
permittypemapped
Building      485134
Demolition     17680

permitclassmapped
permitclassmapped
Residential        413578
Non-Residential     84904
Unspecified          4332

workclassmapped
workclassmapped
Existing    252118
New         245941
              4755

permitclassgroup
permitclassgroup
Single Family           273258
Garage                   64418
Commercial               62472
Two Family               28406
Secondary Suites         18523
Unspecified              13686
Townhouse                10889
Institutional             9695
Apartment                 7002
Industrial                5596
Special Function Ten      3521
General                   2492
Additional Dwelling       1152
Government                 980
Swimmin

`permittype` has 5 values (Residential Improvement Project 226,823; Single Construction Permit 156,507; Commercial / Multi Family Project 101,057; Demolition 17,680; Environmental Restoration Permit 747) and no blanks. `permittypemapped` has 2 values: Building (485,134) and Demolition (17,680). `permitclassmapped` has 3 values: Residential (413,578), Non-Residential (84,904) and Unspecified (4,332). `workclassmapped` has 2 values, Existing (252,118) and New (245,941), and 4,755 blanks. `permitclassgroup` has 16 values, led by Single Family (273,258), Garage (64,418) and Commercial (62,472), and includes Unspecified (13,686) and Special Function Ten (3,521).

**Decision: classification levels.** Each question uses the level of classification in the table below. All classification columns are kept as recorded, and only one clean column is created.

| Question | Main level | Second view |
|---|---|---|
| Q1 Time to issue | `permittype` (5 values) | `permitclassmapped` |
| Q2 Project cost | `permitclassgroup` (16 values) | none |
| Q3 Volume over time | `permittypemapped` (2 values) | `permitclassmapped` |
| Q4 Geography | `permitclassmapped` (3 values) | none |
| Q5 Missingness | `permittype` (5 values) | none |

**Doubtful values and reasons:**
- **`Unspecified`** (4,332 rows in `permitclassmapped`, 13,686 in `permitclassgroup`) is kept as its own category and reported in every table. Merging it into another class would be a guess (the 13,686 split 9,304 Residential, 4,332 Unspecified and 50 Non-Residential in the other column), and removing it would lose 2.7% of the applications.
- **The 4,755 blanks in `workclassmapped`** are a real missing value. They are set to missing in the new column `workclassmapped_clean`, so Q5 can count them and no analysis treats empty text as a category by accident. The raw column is unchanged.
- **`Special Function Ten`** (3,521 rows, 0.7%) is not defined by the portal. It is kept as recorded, as its own category, and the data dictionary and the tables state that its meaning is not published. Merging it with another class would be a guess.

In [ ]:
# Clean column: same as the raw one, but blanks become missing
df["workclassmapped_clean"] = df["workclassmapped"].replace("", pd.NA)

# Check the result
print(df["workclassmapped_clean"].value_counts(dropna=False).to_string())


workclassmapped_clean
Existing    252118
New         245941
<NA>          4755


`workclassmapped_clean` has 252,118 Existing, 245,941 New and 4,755 missing values. The other classification columns are unchanged, and `Unspecified` and `Special Function Ten` stay as their own categories.

### 1.3.6 Summary of the derived variables

| Step | Variable | Definition | Empty or flagged rows |
|---|---|---|---|
| 1.3.1 | `status_group` | The 24 statuses in six groups, from the dates of each status | 0 empty |
| 1.3.1 | `status_date_conflict` | Marks dates that contradict the status group | 205 flagged |
| 1.3.2 | `days_to_issue` | Days from application to issue | 18,864 empty |
| 1.3.2 | `issued_same_day` | True when `days_to_issue` is 0 | 18,864 empty |
| 1.3.3 | `days_to_close` | Days from issue to completion | 33,331 empty |
| 1.3.4 | `application_year`, `application_month`, `application_yearmonth` | Parts of the application date | 0 empty |
| 1.3.4 | `full_year`, `full_month` | Complete periods of the application date | 27,403 and 2,089 outside |
| 1.3.5 | `workclassmapped_clean` | Work class with blanks as missing | 4,755 empty |

The survival variables for the Kaplan-Meier curves of Q1 (event indicator and follow-up time) are not created here. They are defined in Part B, when the method is chosen, from `applieddate`, `issueddate`, `completeddate`, `status_group` and the flags of section 1.2.

In [ ]:
# Check the working table: snapshot columns plus all columns created in sections 1.2 and 1.3
new_cols = [c for c in df.columns if c not in snap.columns]
print("Shape:", df.shape)
print("New columns:", len(new_cols))
print(new_cols)
print("Rows:", len(df), "| unique permit numbers:", df["permitnum"].nunique())

Shape: (502814, 46)
New columns: 25
['date_order_issue', 'issue_date_missing_closed', 'completion_date_missing_completed', 'estprojectcost_clean', 'cost_issue', 'totalsqft_clean', 'sqft_issue', 'housingunits_clean', 'housingunits_issue', 'location_missing', 'latitude_clean', 'longitude_clean', 'communityname_clean', 'communitycode_clean', 'status_group', 'status_date_conflict', 'days_to_issue', 'issued_same_day', 'days_to_close', 'application_year', 'application_month', 'application_yearmonth', 'full_year', 'full_month', 'workclassmapped_clean']
Rows: 502814 | unique permit numbers: 502814


The working table has 502,814 rows, one for each permit number, and 46 columns: the 21 columns of the snapshot and 25 new columns (14 from the handling decisions of section 1.2 and 11 derived variables). No row was removed.



## 1.4 Processed dataset

The processed dataset is the working table saved as a file. It is built only from the snapshot and the rules of sections 1.2 and 1.3, so it can be rebuilt at any time.



### 1.4.1 Columns

**Decision.** All 46 columns are kept. Each raw column stays next to its clean version and its flag, so anyone can see what was changed and every decision stays reversible. The columns are ordered in groups: identifier and status, dates and durations, time variables, classification, cost and size, and location.

In [ ]:
# Columns of the processed dataset, in groups
PROCESSED_COLS = [
    # identifier and status
    "permitnum", "statuscurrent", "status_group", "status_date_conflict",
    # dates and their checks
    "applieddate", "issueddate", "completeddate",
    "date_order_issue", "issue_date_missing_closed", "completion_date_missing_completed",
    # durations
    "days_to_issue", "issued_same_day", "days_to_close",
    # time variables
    "application_year", "application_month", "application_yearmonth", "full_year", "full_month",
    # classification
    "permittype", "permittypemapped", "permitclass", "permitclassgroup", "permitclassmapped",
    "workclass", "workclassgroup", "workclassmapped", "workclassmapped_clean",
    # cost and size
    "estprojectcost", "estprojectcost_clean", "cost_issue",
    "totalsqft", "totalsqft_clean", "sqft_issue",
    "housingunits", "housingunits_clean", "housingunits_issue",
    # location
    "communitycode", "communitycode_clean", "communityname", "communityname_clean",
    "latitude", "latitude_clean", "longitude", "longitude_clean", "location_missing", "locationcount",
]

# Check that the list has the same columns as the working table, with no repeats
assert len(PROCESSED_COLS) == len(set(PROCESSED_COLS)) == 46
assert set(PROCESSED_COLS) == set(df.columns)

# Build the processed table in this order
processed = df[PROCESSED_COLS].copy()
print(processed.shape)

(502814, 46)


The processed table has the 46 columns in the chosen order and all 502,814 rows.

### 1.4.2 File format

The processed table has typed columns (dates, numbers, True/False values, and integers with missing values). Two formats are compared by size and by how well they keep these types.

In [ ]:
# Save the processed table in two formats in a temporary folder to compare them
import os
processed.to_csv("test_processed.csv.gz", index=False)         # compressed CSV, types are lost
processed.to_parquet("test_processed.parquet", index=False)    # Parquet, types are kept

# Show the size of each file in megabytes
for f in ["test_processed.csv.gz", "test_processed.parquet"]:
    print(f"{f}: {os.path.getsize(f) / 1_000_000:.1f} MB")

# Read the Parquet file back and check that the types are kept
back = pd.read_parquet("test_processed.parquet")
print("\nSame shape:", back.shape == processed.shape)
print("Same types:", (back.dtypes == processed.dtypes).all())

test_processed.csv.gz: 27.9 MB
test_processed.parquet: 30.9 MB

Same shape: True
Same types: True


Both files are over the 25 MB limit of the GitHub web upload. Parquet with stronger compression is tested next.

In [ ]:
# Try stronger compression in Parquet (the data does not change)
for codec in ["zstd", "brotli"]:
    name = f"test_processed_{codec}.parquet"
    processed.to_parquet(name, index=False, compression=codec)
    print(f"{codec}: {os.path.getsize(name) / 1_000_000:.1f} MB")

zstd: 24.9 MB
brotli: 22.4 MB


Parquet with brotli compression (22.4 MB) and with zstd (24.9 MB) are both under the 25 MB limit of the GitHub web upload.

**Decision.** The processed dataset is saved as Parquet with brotli compression, with the name `building_permits_processed_2026-10-27.parquet` in `data/processed/`.

**Reasons:**
- Parquet keeps the types of the columns (dates, numbers, True/False values and integers with missing values), which a CSV file loses.
- Brotli gives the smallest file (22.4 MB), under the 25 MB limit of the GitHub web upload and with a larger margin than zstd (24.9 MB).
- Python reads it with no extra step, and R reads it with the `arrow` package.
- The date in the name is the date of the snapshot it was built from.

In [ ]:
# Date of the snapshot, used in the names of the files
SNAP_DATE = "2026-10-27"

# Final file name, with the date of the snapshot it was built from
PROCESSED_FILE = PROCESSED_DIR / f"building_permits_processed_{SNAP_DATE}.parquet"

# Save the processed table
processed.to_parquet(PROCESSED_FILE, index=False, compression="brotli")

# Remove the test files used for the size comparison
for f in ["test_processed.csv.gz", "test_processed.parquet",
          "test_processed_zstd.parquet", "test_processed_brotli.parquet"]:
    os.remove(f)

# Show the saved file and its size
print(PROCESSED_FILE, f"{os.path.getsize(PROCESSED_FILE) / 1_000_000:.1f} MB")

# Read it back and compare it with the table in memory
back = pd.read_parquet(PROCESSED_FILE)
print("Same shape:  ", back.shape == processed.shape)
print("Same types:  ", (back.dtypes == processed.dtypes).all())
print("Same content:", back.equals(processed))
print("Same missing counts:", (back.isna().sum() == processed.isna().sum()).all())

data/processed/building_permits_processed_2026-10-27.parquet 22.4 MB
Same shape:   True
Same types:   True
Same content: False
Same missing counts: True


In [ ]:
# Find the columns that differ after reading the file back, and compare only their non-missing values
for col in processed.columns:
    a, b = processed[col], back[col]
    if not a.equals(b):
        both = a.notna() & b.notna()
        n_diff = int((a[both] != b[both]).sum())
        print(f"{col} | type: {a.dtype} | differing non-missing values: {n_diff}")

workclassmapped_clean | type: object | differing non-missing values: 0


The saved file has the same shape, the same types and the same number of missing values in every column as the table in memory. Only one column, `workclassmapped_clean`, is not reported as equal, because its missing values are stored as `pd.NA` in memory and read back as `None` from the file. Its non-missing values are all identical, so the file is a faithful copy.

### 1.4.3 Storing the processed dataset in the repository

The file is downloaded from Colab to the local computer and uploaded to the repository folder `data/processed/` on GitHub.

In [ ]:
# Download the processed dataset from Colab to the local computer
from google.colab import files
files.download(str(PROCESSED_FILE))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 1.4.4 Check of the stored file

The processed dataset is read from the repository on GitHub, the same way later notebooks will read it, and compared with the table in memory.

In [ ]:
import io
# Address of the processed dataset in the public repository (raw file)
PROCESSED_URL = ("https://raw.githubusercontent.com/Emelygabrielaa/"
                 "calgary-building-permits-eda/main/data/processed/"
                 f"building_permits_processed_{SNAP_DATE}.parquet")

# Download the file and read it as a table
r = requests.get(PROCESSED_URL, timeout=300)
r.raise_for_status()                                   # stop with an error if the download fails
from_github_p = pd.read_parquet(io.BytesIO(r.content))

# Compare it with the table in memory
print("Same shape:         ", from_github_p.shape == processed.shape)
print("Same types:         ", (from_github_p.dtypes == processed.dtypes).all())
print("Same missing counts:", (from_github_p.isna().sum() == processed.isna().sum()).all())
print("Unique permit numbers:", from_github_p["permitnum"].nunique())

Same shape:          True
Same types:          True
Same missing counts: True
Unique permit numbers: 502814


The processed dataset read from GitHub has the same shape, the same types and the same number of missing values in every column as the table in memory, and it has 502,814 unique permit numbers. Section 1.4 is complete: the processed dataset is built, verified and stored in the repository.

## 1.5 Data dictionary of the processed dataset

The processed dataset has 46 columns: the 21 columns of the snapshot (described in the Phase 0 notebook) and 25 columns created in Phase 1. The new columns are described here. In each case the raw column is kept unchanged next to its clean version.



### 1.5.1 Columns from the handling decisions (section 1.2)

| Column | Type | Definition | Values and notes |
|---|---|---|---|
| `date_order_issue` | text | Marks applications with dates in an impossible order | none (502,781); completed before issued (31); issued before applied (1); completed before applied (1) |
| `issue_date_missing_closed` | True/False | Closed application (Completed, Expired or File Closed) with no issue date | 292 True |
| `completion_date_missing_completed` | True/False | Completed application with no completion date | 25 True |
| `estprojectcost_clean` | number | Estimated project cost, with negative values and values below 10 set to missing | 39,540 missing, of which 39,023 were already missing in the raw column |
| `cost_issue` | text | Marks the cost problems | none (502,297); below 10 (509); negative (8) |
| `totalsqft_clean` | number | Total square feet, with zeros, negatives and values below 100 set to missing | 346,975 missing, of which 346,307 are structural (applications that are not Single Construction Permit) |
| `sqft_issue` | text | Marks the problems of `totalsqft` | none (502,146); zero (652); below 100 (14); negative (2) |
| `housingunits_clean` | integer | Housing units, with negative values set to missing | 2 missing. Zeros are kept as recorded |
| `housingunits_issue` | text | Marks the problems of `housingunits` | none (500,360); zero in new construction (2,452); negative (2) |
| `location_missing` | True/False | Application without a location (coordinates equal to zero and community "N/A") | 921 True |
| `latitude_clean` | number | Latitude, missing when the location is missing | 921 missing |
| `longitude_clean` | number | Longitude, missing when the location is missing | 921 missing |
| `communityname_clean` | text | Community name, missing when the location is missing | 921 missing |
| `communitycode_clean` | text | Community code, missing when the location is missing | 921 missing |

### 1.5.2 Derived variables (section 1.3)

| Column | Type | Definition | Values and notes |
|---|---|---|---|
| `status_group` | text | The 24 values of `statuscurrent` in six groups, decided from the dates of each status and not from an official definition | Completed (455,111); Cancelled or refused (15,760); Issued, in progress (14,277); Closed after issue (12,079); Open before issue (3,552); On hold (2,035) |
| `status_date_conflict` | text | Marks applications whose dates contradict their status group | none (502,609); issue date in open or hold status (184); completion date in issued status (19); completion date in open or hold status (2) |
| `days_to_issue` | number | Whole days from the application date to the issue date | 18,864 empty (no issue date, or issued before applied). Mean 24.0, median 14 |
| `issued_same_day` | True/False | True when `days_to_issue` is 0 | 105,304 True; 378,646 False; 18,864 empty |
| `days_to_close` | number | Whole days from the issue date to the completion date. The completion date is the date on which the application was closed, whatever the outcome | 33,331 empty (missing date, or completed before issued). Median 160 |
| `application_year` | integer | Year of the application date | 1999 to 2026 |
| `application_month` | integer | Month of the application date | 1 to 12 |
| `application_yearmonth` | text | Year and month of the application date, in the form "2019-03" | 331 distinct values |
| `full_year` | True/False | True for applications in the complete years 2000 to 2025 | 475,411 True; 27,403 False (1999 and 2026) |
| `full_month` | True/False | True for applications from July 1999 to September 2026, the complete months | 2,089 False (June 1999: 390; October 2026: 1,699) |
| `workclassmapped_clean` | text | `workclassmapped` with blanks set to missing | Existing (252,118); New (245,941); 4,755 missing |

In [ ]:
# Number of distinct year-month values in the processed table
print(processed["application_yearmonth"].nunique())

329


## 1.6 Closing summary

### What Phase 1 produced

| File | Content | Size |
|---|---|---|
| `data/raw/building_permits_2026-10-27.csv.gz` | Snapshot of the portal table as received: 502,814 rows, 21 columns, all as text | 20.2 MB |
| `data/processed/building_permits_processed_2026-10-27.parquet` | Processed dataset: 502,814 rows, 46 columns (21 from the snapshot and 25 created in Phase 1), with the types of each column | 22.4 MB |

Both files were checked after being stored in the repository. All later phases read the processed dataset and do not query the portal again. No row was removed in Phase 1.




### What the next phases must remember

- **Clean and raw columns.** The analysis uses the `_clean` columns. The raw columns stay in the file for audit, and each decision can be reversed with them.
- **Time to issue (Q1).** The share of same-day issues is reported separately. It changed sharply over time (24% to 41% until 2017, about 1% since 2021), so the time to issue is analysed by permit type and by period, and not pooled across all years. The survival variables for the Kaplan-Meier curves are defined in Part B.
- **Volume over time (Q3).** Annual counts use `full_year` (2000 to 2025) and monthly counts use `full_month` (July 1999 to September 2026).
- **Project cost (Q2).** The costs between 10 and 100 (121 values) and the extremely high costs are reviewed in Part A as outliers. The currency is not stated by the portal.
- **Geography (Q4).** The 921 applications without a location are not mapped, and the flag `location_missing` marks them.
- **Missingness (Q5).** The flags and the missing values created in Phase 1 are the material for this question. The 921 applications without a location are almost all Cancelled (96.5%).
- **Outliers to review in Part A.** The longest time to issue (4,684 days), the longest time to close (6,768 days), `totalsqft` above 10,000, and the largest costs.
- **Sensitivity checks in Part B.** The main tests are repeated without the rows flagged in `date_order_issue`, `status_date_conflict`, `issue_date_missing_closed` and `housingunits_issue`, to confirm that they do not change the results.
- **Limits of the definitions.** The status groups are based on the dates of each status and not on an official definition. `Special Function Ten` (3,521 applications) is not defined by the portal and is kept as its own category.

### Next phase

Phase 2 (analysis) has one notebook per question. Each one starts from the processed dataset, with the five questions and the classification levels fixed in section 1.3.5, and has Part A (descriptive analysis) followed by Part B (formal tests and models).

-----